# 🦀 Day 2: Separating Modules into Files

Today: how real Rust projects organize code across multiple files.

⚠️ **Note:** File-based modules can't be demonstrated in Jupyter.
This notebook teaches the concepts and conventions — practice by creating a Cargo project!

---

## 📂 Module File Layout

Rust has two conventions for organizing modules into files:

### Convention 1: Single file per module
```
src/
├── main.rs         // crate root
├── math.rs          // mod math
├── utils.rs         // mod utils
└── models.rs        // mod models
```

**In `main.rs`:**
```rust
mod math;     // Looks for src/math.rs
mod utils;    // Looks for src/utils.rs
mod models;   // Looks for src/models.rs

fn main() {
    let sum = math::add(3, 4);
    println!("Sum: {}", sum);
}
```

**In `math.rs`:**
```rust
pub fn add(a: i32, b: i32) -> i32 {
    a + b
}

pub fn subtract(a: i32, b: i32) -> i32 {
    a - b
}
```

### Convention 2: Directory with `mod.rs` (for sub-modules)
```
src/
├── main.rs
└── models/
    ├── mod.rs        // mod models (entry point)
    ├── user.rs       // models::user
    └── product.rs    // models::product
```

**In `main.rs`:**
```rust
mod models;  // Looks for src/models/mod.rs

fn main() {
    let user = models::user::User::new("Alice");
    // Or with re-export:
    let user = models::User::new("Alice");
}
```

**In `models/mod.rs`:**
```rust
pub mod user;
pub mod product;

// Re-export for convenience
pub use user::User;
pub use product::Product;
```

**In `models/user.rs`:**
```rust
pub struct User {
    pub name: String,
    pub email: String,
}

impl User {
    pub fn new(name: &str) -> Self {
        User {
            name: name.to_string(),
            email: format!("{}@example.com", name.to_lowercase()),
        }
    }
}
```

### Convention 3: Modern Rust (2018+) — file as directory
```
src/
├── main.rs
├── models.rs         // mod models (entry point)
└── models/
    ├── user.rs        // models::user
    └── product.rs     // models::product
```

**In `models.rs`:**
```rust
pub mod user;
pub mod product;

pub use user::User;
pub use product::Product;
```

This is the **recommended** approach in modern Rust.

## 📦 `lib.rs` vs `main.rs`

A crate can have:
- `src/main.rs` — binary crate (creates an executable)
- `src/lib.rs` — library crate (creates a library)
- **Both!** — binary uses the library

```
src/
├── main.rs     // Binary: uses the library
├── lib.rs      // Library: exports the API
├── math.rs
└── utils.rs
```

**In `lib.rs`:**
```rust
pub mod math;
pub mod utils;
```

**In `main.rs`:**
```rust
use my_crate::math;  // Import from the library by crate name
use my_crate::utils;

fn main() {
    println!("{}", math::add(1, 2));
}
```

## 🎯 Full Example Project

Let's walk through a complete project structure:

```
my_calculator/
├── Cargo.toml
└── src/
    ├── main.rs
    ├── lib.rs
    ├── operations.rs
    ├── operations/
    │   ├── basic.rs
    │   └── advanced.rs
    └── display.rs
```

In [ ]:
// Let's simulate this structure inline to understand the concepts

// === operations/basic.rs ===
mod operations {
    pub mod basic {
        pub fn add(a: f64, b: f64) -> f64 { a + b }
        pub fn subtract(a: f64, b: f64) -> f64 { a - b }
        pub fn multiply(a: f64, b: f64) -> f64 { a * b }
        pub fn divide(a: f64, b: f64) -> Result<f64, String> {
            if b == 0.0 { Err("Division by zero".into()) }
            else { Ok(a / b) }
        }
    }

    pub mod advanced {
        pub fn power(base: f64, exp: f64) -> f64 { base.powf(exp) }
        pub fn sqrt(x: f64) -> Result<f64, String> {
            if x < 0.0 { Err("Cannot sqrt negative".into()) }
            else { Ok(x.sqrt()) }
        }
    }

    // Re-export common items
    pub use basic::{add, subtract, multiply, divide};
    pub use advanced::{power, sqrt};
}

// === display.rs ===
mod display {
    pub fn print_result(a: f64, op: &str, b: f64, result: f64) {
        println!("{} {} {} = {:.4}", a, op, b, result);
    }
}

// === main.rs ===
use operations::*;
use display::print_result;

print_result(10.0, "+", 5.0, add(10.0, 5.0));
print_result(10.0, "-", 5.0, subtract(10.0, 5.0));
print_result(10.0, "×", 5.0, multiply(10.0, 5.0));
print_result(2.0, "^", 10.0, power(2.0, 10.0));

match divide(10.0, 3.0) {
    Ok(r) => print_result(10.0, "÷", 3.0, r),
    Err(e) => println!("Error: {}", e),
}

## 🏋️ Practice Project

Create this project structure on your machine:

```bash
cargo new grade_organizer
cd grade_organizer
```

Then create:
1. `src/models.rs` — `Student` and `Grade` types
2. `src/operations.rs` — average, highest, lowest functions
3. `src/display.rs` — pretty-print report
4. `src/lib.rs` — export all modules
5. `src/main.rs` — use the library

This is yesterday's grade system refactored into proper modules!

## 🎯 Key Takeaways

1. `mod name;` in a file tells Rust to look for `name.rs` or `name/mod.rs`
2. Modern Rust prefers `name.rs` + `name/` directory over `name/mod.rs`
3. `lib.rs` defines a library crate; `main.rs` defines a binary crate
4. `pub use` re-exports items for cleaner public APIs
5. Always keep `mod` declarations in parent files

---
**Tomorrow:** External crates and crates.io! 📦